# Практическое занятие 2. Анализ модели 5V Big Data на примере различных наборов данных

**Технологии больших данных (Big Data)**, 6 семестр.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/p02.ipynb)

**Цели занятия:**

- Понять, что означает каждая из пяти характеристик Big Data: Volume, Velocity, Variety, Veracity, Value
- Научиться измерять характеристики 5V с помощью `pandas`: размер, скорость поступления, разнообразие, качество
- Оценить реальные наборы данных по модели 5V и решить, какие из них действительно относятся к Big Data

> Автоматическая проверка заданий доступна в веб-тренажёре. В блокноте выполняйте код и сверяйте результат с условием.

In [ ]:
# ▶ Подготовка среды: выполните эту ячейку первой.
# Загружает учебные данные из репозитория тренажёра и запускает локальный веб-сервер
# (нужен для заданий с requests: учебный сайт SITE и учебный API).
import os, sys, subprocess, time

REPO = "https://github.com/OWNER/REPO.git"
if not os.path.exists("data/manifest.json"):
    if "OWNER/REPO" in REPO:
        raise SystemExit("Укажите в REPO адрес репозитория тренажёра на GitHub, например https://github.com/<логин>/bigdata-trainer.git")
    if not os.path.exists("bigdata-trainer"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, "bigdata-trainer"], check=True)
    os.chdir("bigdata-trainer")

subprocess.Popen([sys.executable, "-m", "http.server", "8765"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
BASE_URL = "http://localhost:8765/"
SITE = BASE_URL + "data/site/"
API = BASE_URL + "data/api/v1/"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "wordcloud", "seaborn"])
print("Готово. Рабочая папка:", os.getcwd())

## Теория

## Модель 5V

«Большие данные» — это не просто «много данных». Набор данных относят к Big Data, когда обычные инструменты (Excel, одна база данных на ноутбуке) перестают справляться с ним сразу по нескольким причинам. Эти причины описывает модель **5V**:

| V | Вопрос | Как измерить | Инструмент pandas |
|---|---|---|---|
| **Volume** (объём) | Сколько данных? | байты, строки, рост за год | `os.path.getsize`, `len(df)`, `df.memory_usage(deep=True)` |
| **Velocity** (скорость) | Как быстро поступают? | записей в секунду/минуту, пиковая нагрузка | `pd.to_datetime`, `resample`, `dt.floor` |
| **Variety** (разнообразие) | Насколько они разные? | число форматов, источников, языков, платформ | `value_counts`, `nunique`, `str.split` |
| **Veracity** (достоверность) | Можно ли им доверять? | % пропусков, дубликатов, ошибок | `isna().mean()`, `duplicated()` |
| **Value** (ценность) | Какую пользу можно извлечь? | ответы на бизнес-вопросы | `groupby`, `str.findall`, сводные таблицы |

Иногда добавляют ещё **Variability** (изменчивость смысла и формата) и **Visualization**, получая модель 7V, но основу составляют пять перечисленных.

> **Заметка: Big Data — это не только размер**  
> Поток постов из Telegram за полчаса весит меньше мегабайта, но поступает непрерывно, на трёх языках и в двух алфавитах, с дублями и пропусками. Для компьютерного лингвиста такой поток — типичная Big Data-задача: важны скорость и разнообразие, а не только объём.

## Volume: объём

```python
import os
import pandas as pd

size_bytes = os.path.getsize("data/tweets_stream.csv")   # размер файла на диске
tweets = pd.read_csv("data/tweets_stream.csv")
mem = tweets.memory_usage(deep=True).sum()              # память в ОЗУ, байт
print(size_bytes / 1024, "КБ на диске;", mem / 1024, "КБ в памяти")
```

Параметр `deep=True` обязателен: без него pandas не учитывает реальную длину строк, и объём текстовых столбцов сильно занижается. Единицы: 1 КБ = 1024 байта, 1 МБ = 1024 КБ, 1 ГБ = 1024 МБ, 1 ТБ = 1024 ГБ.

Объём полезно **экстраполировать**: если за 30 минут пришло `N` постов, то за сутки придёт `N * 2 * 24`, а за год — `N * 2 * 24 * 365`.

## Velocity: скорость

```python
tweets["timestamp"] = pd.to_datetime(tweets["timestamp"])
duration = tweets["timestamp"].max() - tweets["timestamp"].min()   # Timedelta
per_second = len(tweets) / duration.total_seconds()

# число постов в каждую минуту — два равноценных способа
per_minute = tweets.set_index("timestamp").resample("1min").size()
per_minute = tweets["timestamp"].dt.floor("min").value_counts().sort_index()
per_minute.max(), per_minute.idxmax()                              # пик нагрузки
```

Систему проектируют под **пиковую**, а не среднюю нагрузку: если в праздник Навруз число постов выросло в 10 раз, сервер не должен «упасть».

## Variety: разнообразие

```python
tweets["platform"].value_counts(normalize=True) * 100   # доли платформ в %
tweets["lang"].nunique()                                # число языков
tweets["text"].str.contains("[А-Яа-яЁё]", na=False)      # текст на кириллице?

catalog = pd.read_csv("data/datasets_catalog.csv")
catalog["formats"].str.split(", ")                      # "CSV, Parquet" -> ["CSV", "Parquet"]
```

Для узбекского языка разнообразие особенно заметно: одни и те же слова пишут латиницей и кириллицей, апостроф в *oʻ, gʻ* набирают пятью разными символами.

## Veracity: достоверность

```python
tweets.isna().mean() * 100        # % пропусков в каждом столбце
tweets.duplicated().sum()         # число полностью повторяющихся строк
clean = tweets.drop_duplicates().dropna(subset=["text"])
```

Пропуски и дубликаты искажают статистику: повторённый пост «накручивает» популярность хештега, а пустой текст бесполезен для лингвистического анализа.

## Value: ценность

Ценность — это ответ на конкретный вопрос: какие темы обсуждают, на какой платформе аудитория активнее, из каких городов пишут. Например, хештеги извлекаются регулярным выражением:

```python
tags = clean["text"].str.findall(r"#\w+").explode().value_counts()
tags.head()      # самые популярные хештеги
```

> **Совет: Каталог наборов данных**  
> Файл `data/datasets_catalog.csv` описывает 10 наборов: от журнала турникетов Ташкентского метро до оценок одной студенческой группы. Столбцы: `size_gb`, `records`, `update_freq`, `formats`, `sources`, `missing_pct`, `duplicates_pct`, `value`. По ним можно сравнить наборы по модели 5V.

## Задание 1. Какая это V?

_Это задание выполняется в веб-тренажёре (вопрос)._

Ниже перечислены факты о потоке постов `data/tweets_stream.csv`. Какие из них относятся к характеристике **Veracity** (достоверность)? Выберите все подходящие варианты.

## Задание 2. Volume: объём потока

Файл `data/tweets_stream.csv` содержит посты из соцсетей за **30 минут**. Оцените его объём:

- `tweets` — DataFrame с данными файла, `n_rows` — число строк;
- `file_kb` — размер файла на диске в килобайтах (`os.path.getsize(...) / 1024`), округлите до 1 знака;
- `mem_kb` — объём DataFrame в памяти в килобайтах (`memory_usage(deep=True).sum() / 1024`), до 1 знака;
- `bytes_per_post` — сколько байт файла приходится в среднем на один пост;
- `year_gb` — прогноз: сколько **гигабайт** накопится за год, если посты будут поступать с той же скоростью (за год пройдёт `2 * 24 * 365` таких получасовых интервалов). Округлите до 2 знаков.

<details><summary>💡 Подсказка</summary>

`os.path.getsize(путь)` возвращает байты. Годовой объём: `bytes_per_post * n_rows * 2 * 24 * 365 / 1024**3`.

</details>

In [ ]:
import os
import pandas as pd

tweets = ...
n_rows = ...
file_kb = ...
mem_kb = ...
bytes_per_post = ...
year_gb = ...

print(f"Постов: {n_rows}, файл: {file_kb} КБ, в памяти: {mem_kb} КБ")
print(f"В среднем {bytes_per_post} байт на пост")
print(f"За год: {year_gb} ГБ; 1000 таких каналов: {year_gb * 1000 / 1024:.1f} ТБ")

## Задание 3. Velocity: скорость поступления

DataFrame `tweets` уже загружен. Измерьте скорость потока:

1. Преобразуйте столбец `timestamp` в тип datetime (`pd.to_datetime`).
2. `duration_min` — длительность наблюдения в минутах (разница между последним и первым постом).
3. `per_second` — среднее число постов в секунду.
4. `per_minute` — Series: сколько постов пришло в **каждую минуту** (индекс — минута, значение — число постов).
5. `peak_count` — максимальное число постов за минуту, `peak_minute` — минута, когда он был достигнут.
6. Постройте линейный график `per_minute` (`per_minute.plot()`).

Обратите внимание: последняя минута (10:30) неполная — в неё попал всего один пост.

<details><summary>💡 Подсказка</summary>

Длительность: `(tweets["timestamp"].max() - tweets["timestamp"].min()).total_seconds()`. Посты по минутам: `tweets.set_index("timestamp").resample("1min").size()`; пик — `max()` и `idxmax()`.

</details>

In [ ]:
# Подготовительный код задания
import pandas as pd
tweets = pd.read_csv("data/tweets_stream.csv")

In [ ]:
import matplotlib.pyplot as plt

tweets["timestamp"] = ...
duration_min = ...
per_second = ...

per_minute = ...
peak_count = ...
peak_minute = ...

print(f"Наблюдение: {duration_min:.1f} мин, {per_second:.2f} поста/с")
print("Пик:", peak_minute, peak_count)

## Задание 4. Variety: платформы, языки, форматы

Загружены `tweets` (поток постов) и `catalog` (каталог из 10 наборов данных). Оцените разнообразие:

- `platform_share` — доля каждой платформы в процентах (Series, сумма ≈ 100);
- `n_platforms` и `n_langs` — число различных платформ и языков;
- `n_cyr` — сколько постов содержат буквы **кириллицы** (регулярное выражение `[А-Яа-яЁё]`, пропуски не считать);
- столбец `catalog["n_formats"]` — сколько форматов указано у каждого набора (форматы перечислены через `", "`);
- `all_formats` — **множество** (set) всех различных форматов, встречающихся в каталоге.

<details><summary>💡 Подсказка</summary>

`value_counts(normalize=True) * 100` даёт проценты. Форматы: `catalog["formats"].str.split(", ")` — списки; длина — `.str.len()`; множество — `set(catalog["formats"].str.split(", ").explode())`.

</details>

In [ ]:
# Подготовительный код задания
import pandas as pd
tweets = pd.read_csv("data/tweets_stream.csv")
catalog = pd.read_csv("data/datasets_catalog.csv")

In [ ]:
platform_share = ...
n_platforms = ...
n_langs = ...
n_cyr = ...

catalog["n_formats"] = ...
all_formats = ...

print(platform_share.round(1))
print("Платформ:", n_platforms, "языков:", n_langs, "постов кириллицей:", n_cyr)
print(len(all_formats), "форматов:", sorted(all_formats))
catalog[["name", "formats", "n_formats"]]

## Задание 5. Veracity: пропуски и дубликаты

Оцените достоверность потока `tweets` и подготовьте чистую версию:

- `missing_pct` — Series: процент пропусков в каждом столбце;
- `n_dups` — число строк, которые полностью повторяют предыдущие;
- `clean` — копия без дубликатов и без постов с пустым `text`; пропуски в `location` замените строкой `"noma'lum"` («неизвестно»);
- `quality` — какой процент исходных строк остался в `clean` (округлите до 1 знака).

<details><summary>💡 Подсказка</summary>

`tweets.isna().mean() * 100`, `tweets.duplicated().sum()`. Цепочка: `tweets.drop_duplicates().dropna(subset=["text"])`, затем `clean["location"] = clean["location"].fillna("noma'lum")`.

</details>

In [ ]:
# Подготовительный код задания
import pandas as pd
tweets = pd.read_csv("data/tweets_stream.csv")

In [ ]:
missing_pct = ...
n_dups = ...

clean = ...

quality = ...

print(missing_pct.round(1))
print("Дубликатов:", n_dups)
print(f"Осталось {len(clean)} из {len(tweets)} строк ({quality}%)")

## Задание 6. Value: что обсуждают и где

Из очищенного потока `clean` извлеките полезные знания:

- `hashtags` — Series: частоты хештегов по убыванию (хештег — `#` и следующие за ним буквы/цифры, регулярное выражение `#\w+`);
- `top_tag` — самый популярный хештег;
- добавьте столбец `engagement = likes + 2 * reposts` (репост ценнее лайка);
- `best_platform` — платформа с наибольшим **средним** `engagement`;
- `top_city` — город (`location`), откуда пришло больше всего постов.

<details><summary>💡 Подсказка</summary>

`clean["text"].str.findall(r"#\w+")` возвращает списки хештегов; `.explode().value_counts()` превращает их в частоты. Среднее по группам: `groupby("platform")["engagement"].mean().idxmax()`.

</details>

In [ ]:
# Подготовительный код задания
import pandas as pd
tweets = pd.read_csv("data/tweets_stream.csv")
clean = tweets.drop_duplicates().dropna(subset=["text"]).copy()

In [ ]:
hashtags = ...
top_tag = ...

clean["engagement"] = ...
best_platform = ...
top_city = ...

print(hashtags.head())
print("Тренд:", top_tag, "| платформа:", best_platform, "| город:", top_city)

## Задание 7. Оценка наборов данных по 5V

Оцените 10 наборов из `catalog` по трём «техническим» V. Добавьте столбцы со значениями 0 или 1:

| Столбец | 1, если… |
|---|---|
| `volume` | `size_gb` не меньше 100 |
| `velocity` | данные обновляются не реже раза в час: `update_freq` — одно из `"real-time"`, `"har 10 daqiqa"`, `"har soat"` |
| `variety` | в `formats` указано два формата или больше |

Затем:

- `score` = `volume + velocity + variety`;
- `big_data` — **список** названий (`name`) наборов с `score` ≥ 2;
- выведите таблицу, отсортированную по `score` по убыванию.

<details><summary>💡 Подсказка</summary>

Условие превращается в 0/1 так: `(catalog["size_gb"] >= 100).astype(int)`. Для частоты — `catalog["update_freq"].isin(fast)`. Список имён: `catalog.loc[catalog["score"] >= 2, "name"].tolist()`.

</details>

In [ ]:
# Подготовительный код задания
import pandas as pd
catalog = pd.read_csv("data/datasets_catalog.csv")

In [ ]:
fast = ["real-time", "har 10 daqiqa", "har soat"]

catalog["volume"] = ...
catalog["velocity"] = ...
catalog["variety"] = ...
catalog["score"] = ...

big_data = ...

print("Big Data:", big_data)
catalog.sort_values("score", ascending=False)[["name", "volume", "velocity", "variety", "score", "value"]]

## Самостоятельная работа

## Самостоятельная работа: «Информационная безопасность и конфиденциальность данных в технологиях Big Data»

1. В `tweets_stream.csv` столбец `user` — это идентификатор человека. Замените его **псевдонимом**: хешем SHA-256 с «солью» (`hashlib.sha256((salt + user).encode()).hexdigest()[:12]`). Проверьте, что число уникальных пользователей не изменилось.
2. Подумайте, какие ещё столбцы могут раскрыть личность (`location`, текст поста). Как их обобщить (город → регион) без потери ценности (Value)?
3. Выберите любой набор из `datasets_catalog.csv` и напишите 5–7 предложений: какие риски конфиденциальности в нём есть и какие меры защиты (шифрование, разграничение доступа, анонимизация) вы бы применили.